# hai07 - ukur kecepatan latih di Google Colab (GPU atau TPU)
Satu notebook, tiga jalur yang dipilih otomatis: **GPU** (T4/L4/A100/H100; `tools/bench_train.py`, satu proses), **TPU dengan `torch_tpu`** (runtime TPU Colab yang terbaru: perangkat bernama `tpu`; `bench_train.py` dijalankan DI DALAM proses notebook ini, karena TPU hanya bisa dipegang satu proses), dan **TPU dengan `torch_xla`** (`tools/bench_train_xla.py`, mode SPMD, proses terpisah).

**Persiapan**
1. Menu **Runtime > Change runtime type**: pilih TPU atau GPU yang tersedia di akun Anda.
2. Unggah `kaggle/hai07-bench.zip` (dibuat oleh `python tools/package_bench_kit.py`, ±0,75 MB; **harus zip terbaru**) ke panel berkas kiri (`/content`), atau ke Google Drive lalu isi `DRIVE_ZIP` di sel 1.
3. Jalankan **sel 1 sampai 3 dulu**. Sel 3 adalah uji asap kecil dan **berhenti** bila gagal; jangan jalankan sel 4 sebelum sel 3 berhasil.

Pada jalur `torch_tpu`: tiap konfigurasi di sel 4 berjalan di proses ini; bila satu konfigurasi gagal karena memori, runtime kadang perlu di-restart (Runtime > Restart session, lalu jalankan sel 1 dan 2 lagi).
Perkiraan sel 4: ±15-45 menit (kompilasi di langkah pertama lambat). Hasil: `/content/bench_colab.jsonl`. Colab dapat memutus sesi kapan saja; unduh hasilnya.
Kode untuk `torch_tpu` baru ditulis tanpa bisa diuji di perangkatnya (hanya CPU): kirim ±15 baris terakhir keluaran bila gagal.

In [ ]:
import glob, os, subprocess, sys, zipfile
DRIVE_ZIP = ""   # contoh: "/content/drive/MyDrive/hai07-bench.zip" (setelah from google.colab import drive; drive.mount("/content/drive"))
WORK = "/content/hai07-bench"; os.makedirs(WORK, exist_ok=True)
cands = [p for p in [DRIVE_ZIP, "/content/hai07-bench.zip"] if p and os.path.exists(p)] + glob.glob("/content/**/hai07-bench*.zip", recursive=True)
if not cands:
    from google.colab import files
    print("Pilih berkas hai07-bench.zip ..."); up = files.upload(); cands = [n for n in up if n.endswith(".zip")]
assert cands, "hai07-bench.zip tidak ditemukan: unggah ke /content atau isi DRIVE_ZIP"
zipfile.ZipFile(cands[0]).extractall(WORK)
os.chdir(WORK)
assert os.path.exists("tools/bench_train_xla.py"), "zip lama: unggah hai07-bench.zip terbaru"
assert "peak-tflops" in open("tools/bench_train.py", encoding="utf-8").read(), "zip lama (bench_train.py tanpa --peak-tflops): unggah hai07-bench.zip terbaru"
print("berkas siap di", WORK, "dari", cands[0])


In [ ]:
import importlib.util, subprocess, sys, os
INSTALL_XLA = False   # ubah ke True hanya bila runtime TPU tanpa torch_tpu dan tanpa torch_xla (memasang dapat mengganti versi torch)
import torch
print("torch", torch.__version__)
MODE = None
if torch.cuda.is_available():
    MODE = "gpu"; print("GPU:", torch.cuda.get_device_name(0), "| memori", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
elif importlib.util.find_spec("torch_tpu") is not None:
    MODE = "tpu_torch"
    x = torch.ones(2, 2, device=torch.device("tpu")); print("torch_tpu aktif; uji kecil di perangkat 'tpu':", (x @ x).cpu().tolist())
else:
    if importlib.util.find_spec("torch_xla") is None and INSTALL_XLA:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_xla[tpu]", "-f", "https://storage.googleapis.com/libtpu-releases/index.html"], check=False)
    if importlib.util.find_spec("torch_xla") is not None:
        p = subprocess.run([sys.executable, "-c", "import torch_xla.core.xla_model as xm, torch_xla.runtime as xr; print(xm.xla_device(), xr.global_runtime_device_count())"], capture_output=True, text=True, timeout=300)
        print(p.stdout.strip() or p.stderr.strip()[-600:])
        if p.returncode == 0:
            MODE = "tpu"
print("MODE =", MODE)
assert MODE, "tidak ada GPU maupun TPU yang bisa dipakai: pilih akselerator di Runtime > Change runtime type"


In [ ]:
TPU_PEAK = 197.0   # puncak bf16 TFLOPs per chip, hanya untuk MFU: v5e = 197, v6e = 918
ACCEL = "v5e"      # jalur torch_xla saja: "v5e", "v6e" atau "v2"
import json, os, sys
sys.path[:0] = [os.path.join(WORK, "apps"), os.path.join(WORK, "tools")]
import bench_train as BT

def run_inproc(argv):
    """Jalankan satu konfigurasi di proses ini (jalur torch_tpu); galat apa pun dikembalikan sebagai {'galat': ...}."""
    try:
        return BT.run(BT.parse_args(argv))
    except Exception as e:
        return {"galat": f"{type(e).__name__}: {str(e).splitlines()[0][:200] if str(e) else ''}"}

TPU_ARGS = ["--device", "tpu", "--amp", "--peak-tflops", str(TPU_PEAK), "--device-name", "TPU (torch_tpu)"]
if MODE == "tpu_torch":
    res = run_inproc(["--preset", "tiny", "--ctx", "64", "--batch", "2", "--steps", "3", "--warmup", "2"] + TPU_ARGS)
    print(res)
    ok = "token_per_detik" in res
elif MODE == "tpu":
    p = subprocess.run([sys.executable, "-u", "tools/bench_train_xla.py", "--preset", "tiny", "--ctx", "64", "--batch", "2", "--steps", "3", "--warmup", "2", "--accelerator", ACCEL],
                       capture_output=True, text=True, timeout=900)
    print(p.stdout[-3000:]); print(p.stderr[-4000:])
    ok = any(l.startswith("{") and "token_per_detik" in l for l in p.stdout.splitlines())
else:
    p = subprocess.run([sys.executable, "-u", "tools/bench_train.py", "--preset", "tiny", "--ctx", "64", "--batch", "2", "--steps", "3", "--warmup", "2", "--device", "cuda", "--amp"],
                       capture_output=True, text=True, timeout=900)
    print(p.stdout[-3000:]); print(p.stderr[-4000:])
    ok = any(l.startswith("{") and "token_per_detik" in l for l in p.stdout.splitlines())
assert ok, "UJI ASAP GAGAL: kirim keluaran di atas ke pengembang; jangan lanjut ke sel 4"
print("UJI ASAP BERHASIL (", MODE, ")")


In [ ]:
# (nama, argumen). Daftar ini LANJUTAN: hasil putaran pertama (batch kecil) menunjukkan langkah hampir tidak melambat saat batch naik, jadi dicari batch yang menjenuhkan chip. --batch = per chip.
SMALL = [
    ("100M ctx1024 batch8",       ["--preset", "100m", "--ctx", "1024", "--batch", "8"]),
    ("100M ctx1024 batch16",      ["--preset", "100m", "--ctx", "1024", "--batch", "16"]),
    ("300M ctx1024 ckpt batch8",  ["--preset", "300m", "--ctx", "1024", "--batch", "8", "--grad-checkpoint"]),
    ("300M ctx1024 ckpt batch16", ["--preset", "300m", "--ctx", "1024", "--batch", "16", "--grad-checkpoint"]),
    ("540M ctx1024 ckpt batch4",  ["--preset", "540m", "--ctx", "1024", "--batch", "4", "--grad-checkpoint"]),
    ("540M ctx1024 ckpt batch8",  ["--preset", "540m", "--ctx", "1024", "--batch", "8", "--grad-checkpoint"]),
    ("540M ctx1024 ckpt batch16", ["--preset", "540m", "--ctx", "1024", "--batch", "16", "--grad-checkpoint"]),
    ("540M ctx1024 ckpt batch8 + compile", ["--preset", "540m", "--ctx", "1024", "--batch", "8", "--grad-checkpoint", "--compile"]),
]
COMMON = ["--steps", "8", "--warmup", "3"]
OUT = "/content/bench_colab.jsonl"
if os.path.exists(OUT): os.remove(OUT)
rows = []
failed = set()   # (preset) yang sudah gagal: batch yang lebih besar dilewati (hemat waktu; runtime bisa rusak setelah kehabisan memori)
for name, extra in SMALL:
    print("=" * 90); print(name, flush=True)
    key = extra[1] + ("+compile" if "--compile" in extra else "")
    if MODE == "tpu_torch" and key in failed:
        rows.append({"nama": name, "galat": "dilewati: konfigurasi lebih kecil dengan preset yang sama sudah gagal"}); print("dilewati"); continue
    if MODE == "tpu_torch":
        d = run_inproc(COMMON + TPU_ARGS + extra)
        d["nama"] = name; rows.append(d); print(d)
        if "galat" in d:
            failed.add(key)
        if "galat" not in d:
            open(OUT, "a", encoding="utf-8").write(json.dumps(d, ensure_ascii=False) + "\n")
        continue
    script = "tools/bench_train_xla.py" if MODE == "tpu" else "tools/bench_train.py"
    more = ["--accelerator", ACCEL] if MODE == "tpu" else ["--device", "cuda", "--amp", "--fused-adamw"]
    try:
        p = subprocess.run([sys.executable, "-u", script, "--out", OUT] + COMMON + more + extra, capture_output=True, text=True, timeout=1800)
        last = [l for l in p.stdout.splitlines() if l.startswith("{")]
        if last:
            d = json.loads(last[-1]); d["nama"] = name; rows.append(d); print(last[-1])
        else:
            err = (p.stderr.strip().splitlines() or ["(tanpa keluaran)"])[-1][:200]
            rows.append({"nama": name, "galat": err}); print("GALAT:", err)
    except subprocess.TimeoutExpired:
        rows.append({"nama": name, "galat": "melewati batas waktu 30 menit"}); print("GALAT: batas waktu")


In [ ]:
cols = ["nama", "parameter_juta", "chip", "token_per_detik", "mfu_persen", "memori_puncak_gb", "memori_dipakai_gb", "bf16_murni", "jam_untuk_10_miliar_token", "galat"]
cols = [c for c in cols if any(c in r for r in rows)]
w = {c: max(len(c), *(len(str(r.get(c, "-"))) for r in rows)) for c in cols}
print("  ".join(c.ljust(w[c]) for c in cols)); print("  ".join("-" * w[c] for c in cols))
for r in rows: print("  ".join(str(r.get(c, "-")).ljust(w[c]) for c in cols))
print("\nPerangkat:", next((r.get("perangkat") for r in rows if r.get("perangkat")), MODE))
print("jam_untuk_10_miliar_token x 1,08 = jam latih 540M (10,8 miliar token) bila hasilnya bertahan stabil; Colab memutus sesi, jadi perlu --resume.")
print("bf16_murni=True: autocast tidak tersedia di perangkat itu, jadi seluruh model dihitung dalam bf16 (cukup untuk mengukur kecepatan).")
print("Salin tabel ini dan kirim ke pengembang. Berkas lengkap:", OUT)
